In [1]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/Users/sasha/miniconda3/envs/lima/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("Class:", tokenizer.__class__)
print("Vocab size:", tokenizer.vocab_size)
print("Model max length:", tokenizer.model_max_length)     # The maximum length (in number of tokens) for the inputs to the transformer model.
print("EOS token:", tokenizer.eos_token, tokenizer.eos_token_id)
print("PAD token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("UNK token:", tokenizer.unk_token, tokenizer.unk_token_id)
print("Chat template exists:", tokenizer.chat_template is not None)

Class: <class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>
Vocab size: 151643
Model max length: 131072
EOS token: <|im_end|> 151645
PAD token: <|endoftext|> 151643
UNK token: None None
Chat template exists: True


In [3]:
tokenizer.special_tokens_map

{'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}

In [4]:
tokenizer.all_special_tokens

['<|im_end|>',
 '<|endoftext|>',
 '<|im_start|>',
 '<|object_ref_start|>',
 '<|object_ref_end|>',
 '<|box_start|>',
 '<|box_end|>',
 '<|quad_start|>',
 '<|quad_end|>',
 '<|vision_start|>',
 '<|vision_end|>',
 '<|vision_pad|>',
 '<|image_pad|>',
 '<|video_pad|>']

In [ ]:
text = "The quick brown fox jumps over the lazy dog."
tokens = tokenizer.tokenize(text)
ids = tokenizer.encode(text)
print(tokens, type(tokens))
print(ids, type(ids))
print(tokenizer.decode(ids))

print("_" * 100)

text = " The quick brown fox jumps over the lazy dog."     # Ġ means: this token starts after a space
tokens = tokenizer.tokenize(text)
ids = tokenizer.encode(text)
print(tokens, type(tokens))
print(ids, type(ids))
print(tokenizer.decode(ids))

['The', 'Ġquick', 'Ġbrown', 'Ġfox', 'Ġjumps', 'Ġover', 'Ġthe', 'Ġlazy', 'Ġdog', '.'] <class 'list'>
[785, 3974, 13876, 38835, 34208, 916, 279, 15678, 5562, 13] <class 'list'>
The quick brown fox jumps over the lazy dog.
____________________________________________________________________________________________________
['ĠThe', 'Ġquick', 'Ġbrown', 'Ġfox', 'Ġjumps', 'Ġover', 'Ġthe', 'Ġlazy', 'Ġdog', '.'] <class 'list'>
[576, 3974, 13876, 38835, 34208, 916, 279, 15678, 5562, 13] <class 'list'>
 The quick brown fox jumps over the lazy dog.


In [14]:
examples = [
    "Hello world",
    "Hello, world",
    "hello world",
    " Hello world",
    "fine-tuning",
    "fine tuning",
    "supervised fine-tuning",
]

for x in examples:
    ids = tokenizer.encode(x)
    print("\nTEXT:", repr(x))
    print("TOKENS:", tokenizer.tokenize(x))
    print("IDS:", ids)
    print("COUNT:", len(ids))


TEXT: 'Hello world'
TOKENS: ['Hello', 'Ġworld']
IDS: [9707, 1879]
COUNT: 2

TEXT: 'Hello, world'
TOKENS: ['Hello', ',', 'Ġworld']
IDS: [9707, 11, 1879]
COUNT: 3

TEXT: 'hello world'
TOKENS: ['hello', 'Ġworld']
IDS: [14990, 1879]
COUNT: 2

TEXT: ' Hello world'
TOKENS: ['ĠHello', 'Ġworld']
IDS: [21927, 1879]
COUNT: 2

TEXT: 'fine-tuning'
TOKENS: ['fine', '-t', 'uning']
IDS: [62057, 2385, 37202]
COUNT: 3

TEXT: 'fine tuning'
TOKENS: ['fine', 'Ġtuning']
IDS: [62057, 41338]
COUNT: 2

TEXT: 'supervised fine-tuning'
TOKENS: ['sup', 'ervised', 'Ġfine', '-t', 'uning']
IDS: [12776, 77990, 6915, 2385, 37202]
COUNT: 5


In [15]:
print(tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba C

In [27]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain supervised fine-tuning in simple words."},
]

formatted = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(formatted)
print(type(formatted))

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Explain supervised fine-tuning in simple words.<|im_end|>
<|im_start|>assistant

<class 'str'>


In [24]:
chat = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True     # <|im_start|>assistant
)

print(chat)
print("Number of tokens:", len(chat["input_ids"]))
print(tokenizer.decode(chat["input_ids"]))

{'input_ids': [151644, 8948, 198, 2610, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 840, 20772, 58989, 6915, 2385, 37202, 304, 4285, 4244, 13, 151645, 198, 151644, 77091, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
Number of tokens: 29
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Explain supervised fine-tuning in simple words.<|im_end|>
<|im_start|>assistant



In [ ]:
manual_wrong = """User: Explain supervised fine-tuning in simple words.
Assistant:"""

correct_ids = tokenizer.encode(formatted)
wrong_ids = tokenizer.encode(manual_wrong)

print("Correct template tokens:", len(correct_ids))
print("Wrong manual tokens:", len(wrong_ids))

print("\nCorrect decoded:")
print(tokenizer.decode(correct_ids))

print("\nWrong decoded:")
print(tokenizer.decode(wrong_ids))

Correct template tokens: 29
Wrong manual tokens: 13

Correct decoded:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Explain supervised fine-tuning in simple words.<|im_end|>
<|im_start|>assistant


Wrong decoded:
User: Explain supervised fine-tuning in simple words.
Assistant:
